In [ ]:
#import Pkg; 
# Pkg.add("Revise")
# Pkg.add("ControlSystems")
# Pkg.add("OrdinaryDiffEq")
# Pkg.add("Distributions")
# Pkg.add("ControlSystemsBase")
# Pkg.add("ControlSystemsBase")
# Pkg.add("TensorCast")
# Pkg.add("Plots")
# Pkg.add("ForwardDiff")
# Pkg.add("DifferentialEquations")
#Pkg.add("Convex")
#Pkg.add("SCS")
#Pkg.add("Optim")

In [ ]:
using Revise;
includet("./kernel_module.jl")
includet("./signal_system.jl")
includet("./system_kernel.jl")
using Plots
using LinearAlgebra
using ForwardDiff
using ControlSystems
using ControlSystemsBase: nonlinearity
using OrdinaryDiffEq
using Distributions
using Random: seed!
using .KernelLearning
using .SignalSystem
using .SystemKernel
#using DifferentialEquations
using Convex, SCS, Optim
using DelimitedFiles
seed!(42);

In [ ]:
# sys=Simple_Nonlinear_sys_gen(0.5,1,[],[-5,-3,-2+im,-2-im])
# x0=zeros(4);
sys=Simple_Nonlinear_sys_gen(0.5,1,[],[-5])
sys7=Simple_Nonlinear_sys_gen(0.5,1,[],[-10,-5])
sys6=Simple_Nonlinear_sys_gen2(0.5,5,[],[-5],2)
x0=zeros(4);
# alpha_n(u)=0.02*(u-25)/(1-exp(-(u-25)/9))
# beta_n(u)=-0.002*exp(u-25)/(1-exp((u-25)/9))

# alpha_m(u)=0.182*(u+35)/(1-exp(-(u+35)/9))
# beta_m(u)=-0.124*(u+35)/(1-exp((u+35)/9))

# alpha_h(u)=0.25*exp(-(u+90)/12)
# beta_h(u)=0.25*exp((u+62)/6)/exp((u+90)/12)

# ## Constants
  C_m     = 1       # μF/cm², membrane capacitance
  G_NaMax = 120     # mS/cm², maximum conductivity of Na channel
  G_KMax  = 36      # mS/cm², maximum conductivity of K channel
  G_L     = 0.3     # mS/cm², leak conductivity
  V_r  = 65.1     # mV, resting potential
  V_Na =  50      # mV, Nernst voltage for Na 
  V_K  = -77       # mV, Nernst voltage for K
  V_L  = -54.387   # mV, Nernst voltage for leak

# ## Function Definitions
alpha_n(dV) = (0.1 - 0.01*(V_r+dV))/(exp(1 - 0.1*(V_r+dV))- 1)
beta_n(dV) = 0.125/(exp(0.0125*(V_r+dV)))
alpha_m(dV) = (2.5 - 0.1*(V_r+dV))/(exp(2.5 - 0.1*(V_r+dV))- 1)
beta_m(dV) = 4/(exp((V_r+dV)/18))
alpha_h(dV) = 0.07/(exp(0.05*(V_r+dV)))
beta_h(dV) = 1/(exp(3 - 0.1*(V_r+dV))+ 1)

# Steady states
n_∞(dV) = alpha_n(dV)/(alpha_n(dV)+beta_n(dV))
m_∞(dV) = alpha_m(dV)/(alpha_m(dV)+beta_m(dV))
h_∞(dV) = alpha_h(dV)/(alpha_h(dV)+beta_h(dV))


function nlsim(sys, u, t ; x0=0)
    dt=t[2]-t[1]
    output_y=Array{Float64,2}(undef,(length(sys(x0,u(x0,t[1]))[2]),length(t)))
    output_x=Array{Float64,2}(undef,(length(x0),length(t)))
    output_u=Array{Float64,2}(undef,(length(u(x0,t[1])),length(t)))
    x=copy(x0)
    for i=1:length(t)
        u_temp=u(x,t[i])
        result=sys(x,u_temp)
        output_y[:,i]=copy(result[2])
        output_x[:,i]=copy(x)
        output_u[:,i]=copy(u_temp)
        x=x+result[1].*dt
    end
    return(output_y,t,output_x,output_u)
end

sys2(x,u)=[-5x.+u,[clamp(0.5*x[1], -5, 5)]]

# sys3(x,u)=[(alpha_n(u[1]- V_r)*(1 .-x).-beta_n(u[1]- V_r).*x), G_KMax .*x.^4*(V_K-u[1])]

# sys4(x,u)=[[(alpha_m(u[1]- V_r)*(1-x[1])-beta_m(u[1]- V_r)*x[1]),(alpha_h(u[1]- V_r)*(1 -x[2])-beta_h(u[1]- V_r)*x[2])],[G_NaMax*x[1]^3*x[2]*(V_Na-u[1])]]

sys3(x,u)=[(alpha_n(u[1])*(1 .-x).-beta_n(u[1]).*x), 36 .*x.^4*(u[1]-V_K)]

sys4(x,u)=[[(alpha_m(u[1])*(1-x[1])-beta_m(u[1])*x[1]),(alpha_h(u[1])*(1 -x[2])-beta_h(u[1])*x[2])],[120*x[1]^3*x[2]*(u[1]-V_Na)]]

sys5(x,u)=[0,G_L*(V_L-u[1])]
x0=zeros(1);

In [ ]:
# Injected Current Function
I_inj(t) = 10 * (5 < t < 30)

function HH_model(u,p,t)
    n, m, h, Vm = u 
    
    # Update transfer rate coefficients, n, m, and h
    V_diff = Vm          # difference between the rest voltage and membrane voltage

    result_Na=sys4([m,h],[V_diff])
    result_K=sys3(n,[V_diff])


    dn = result_K[1]
    dm = result_Na[1][1]
    dh = result_Na[1][2]
    
    # Update cell membrane voltage, Vm
    G_K  = G_KMax  * n^4       # Sodium conductance
    G_Na = G_NaMax * h * m^3   # Potasium conductance
    dVm = (I_inj(t) + (V_Na - Vm)*G_Na + (V_K - Vm)*G_K + (V_L - Vm)*G_L)/C_m 

    [dn; dm; dh; dVm]
end

## Run Model:
u0 = [n_∞(0); m_∞(0) ; h_∞(0); -65.1]
tspan = (0.0,50.0)
prob = ODEProblem(HH_model, u0, tspan)
sol = solve(prob, Euler(), dt=0.0001);

In [ ]:
## Plotting
p1 = plot(sol.t, sol[4,:], legend=false, ylabel="Volatge [mV]")
p2 = plot(sol.t, I_inj.(sol.t), legend=false, lc=:red, ylabel="Current")
p3 = plot(sol.t, sol[1:3,:]', label=["n" "m" "h"], legend=:topright, 
          xlabel="Time [ms]", ylabel="Fraction Active")

l = grid(3, 1, heights=[0.4, 0.2 ,0.4])
plot(p1, p2, p3, layout = l, size=(800,500), lw=2)

In [ ]:
output_o=[-65.1]
output_K_o=[0.0]
output_Na_o=[0.0]
x=[-65.1,m_∞(-65.1),h_∞(-65.1),n_∞(-65.1)] #v,m,h,n

#x=[-65.1,0.0,0.0,0.0] #v,m,h,n
input=20
dt=0.01

for i=1:3999
      
    result_Na=sys4(x[2:3],[x[1]])
    result_K=sys3(x[4],[x[1]])
    
    # Update cell membrane voltage, Vm
    dVm = (input -result_Na[2][1] - result_K[2][1]  - (x[1]-V_L)*G_L)/C_m 
    
    push!(output_o,x[1])
    x[1]=x[1]+dVm*dt
    x[2:3]=x[2:3]+result_Na[1]*dt
    x[4]=x[4]+result_K[1]*dt
   
    push!(output_K_o,result_K[2][1])
    push!(output_Na_o,result_Na[2][1])
end

plot_o=plot([output_o output_K_o output_Na_o])


In [ ]:
plot_o=plot([output_o output_K_o output_Na_o])
savefig(plot_o, "HH_model.png")

In [ ]:
output=[-65.1]
output_K=[0.0]
output_Na=[0.0]
x=[-65.1,m_∞(-65.1),h_∞(-65.1),n_∞(-65.1)] #v,m,h,n

#x=[-65.1,0.0,0.0,0.0] #v,m,h,n
input=40
dt=0.01

for i=1:3999
      
    result_Na=sys4(x[2:3],[x[1]])
    result_K=sys3(x[4],[x[1]])
    
    # Update cell membrane voltage, Vm
    dVm = (input -result_Na[2][1] - result_K[2][1]  - (x[1]-V_L)*G_L)/C_m 
    
    push!(output,x[1])
    x[1]=x[1]+dVm*dt
    x[2:3]=x[2:3]+result_Na[1]*dt
    x[4]=x[4]+result_K[1]*dt
   
    push!(output_K,result_K[2][1])
    push!(output_Na,result_Na[2][1])
end

plot_HH=plot([output output_K output_Na])

In [ ]:
savefig(plot_HH, "HH_model_2.png")

In [ ]:
output_2=ones(1,4000)*(-65.1)
output_K=zeros(1,4000)
output_Na=zeros(1,4000)
#x=[-65.1,m_∞(0),h_∞(0),n_∞(0)] #v,m,h,n

x=[-65.1,m_∞(-65.1),h_∞(-65.1)] #v,m,h,n
input=20
dt=0.01

for i=1:3999
    if i<length(fading_weight_K)
        v_past=[ones(1,length(fading_weight_K) - i).*-65.1 copy(output_2[:,1:i])]
    else
        v_past=output_2[:,i - length(fading_weight_K) + 1:i]
    end
    #result_Na=ion_Na(v_past.-V_Na)
    result_Na=sys4(x[2:3],[x[1]])
    result_K=ion_K(v_past.-V_K)
    
    # Update cell membrane voltage, Vm
    dVm = (input -result_Na[2][1] - result_K[1]  - (x[1]-V_L)*G_L)/C_m 
    
    output_2[1,i+1]=x[1]
    x[1]=x[1]+dVm*dt
   
    x[2:3]=x[2:3]+result_Na[1]*dt
    output_K[1,i+1]=result_K[1]
    output_Na[1,i+1]=result_Na[2][1]
end

plot_K=plot([output_2' output_K' output_Na'])
savefig(plot_K,"HH_model_appK.png")
plot_K

In [ ]:
plot([output_o output_2'])

In [ ]:
x = LinRange(0, 40, 4000)
plot(x, output_2', xlabel="time/s", ylabel="mV")

In [ ]:
output_3=ones(1,4000)*(-65.1)
output_K2=zeros(1,4000)
output_Na2=zeros(1,4000)
rest_potential=-65.1
#x=[-65.1,m_∞(0),h_∞(0),n_∞(0)] #v,m,h,n

x=[-65.1,0,0] #v,m,h,n
input=20
dt=0.01

for i=1:3999

    if num_points_Na<num_points_K
        if i<num_points_K
            v_past=[ones(1,num_points_K - i)*rest_potential copy(output_3[:,1:i])]
        else
            v_past=output_3[:,i - length(fading_weight_K) + 1:i]
        end
        v_past_Na=v_past[:,1+(num_points_K-num_points_Na):num_points_K]
    else
        if i<num_points_Na
            v_past_Na=[ones(1,num_points_Na - i)*rest_potential copy(output_3[:,1:i])]
        else
            v_past_Na=output_3[:,i - length(fading_weight_Na) + 1:i]
        end
        v_past=v_past_Na[:,1+(num_points_Na-num_points_K):num_points_Na]
    end


    result_Na=ion_Na(v_past_Na.-V_Na)
    #result_Na=sys4(x[2:3],[x[1]])
    result_K=ion_K(v_past.-V_K)
    
    # Update cell membrane voltage, Vm
    dVm = (input - result_Na[1] - result_K[1]  - (x[1]-V_L)*G_L)/C_m 

    x[1]=x[1]+dVm*dt

    # if i>=1000
    #     output_3[1,i+1]=x[1]
    # else
    #     output_3[1,i+1]=output_o_re[1,i+1]
    #     x[1]=output_o_re[1,i+1]
    # end

    output_3[1,i+1]=x[1]
    

    output_K2[1,i+1]=result_K[1]
    output_Na2[1,i+1]=result_Na[1]
end

plot_Na_K=plot([output_3' output_K2' output_Na2'])
savefig(plot_Na_K,"HH_model_appK_app_Na_test.png")
plot_Na_K

In [ ]:
t_range = LinRange(0, 40, 4000)
summary_plot=plot(t_range,[output_o output_2' output_4' output_3'],label=["original" "approx_K" "approx_Na" "approx_K_Na" ],xlabel="time/s", ylabel="mV")
savefig(summary_plot,"HH_model_summary.png")
summary_plot

In [ ]:
x=[-65.1,m_∞(-65.1),h_∞(-65.1),n_∞(-65.1)] #v,m,h,n
output_4=ones(1,4000)*(-65.1)
output_K3=zeros(1,4000)
output_Na3=zeros(1,4000)
rest_potential=-65.1
#x=[-65.1,m_∞(0),h_∞(0),n_∞(0)] #v,m,h,n
input=20
dt=0.01

for i=1:3999

    if num_points_Na<num_points_K
        if i<num_points_K
            v_past=[ones(1,num_points_K - i)*rest_potential copy(output_4[:,1:i])]
        else
            v_past=output_4[:,i - length(fading_weight_K) + 1:i]
        end
        v_past_Na=v_past[:,1+(num_points_K-num_points_Na):num_points_K]
    else
        if i<num_points_Na
            v_past_Na=[ones(1,num_points_Na - i)*rest_potential copy(output_4[:,1:i])]
        else
            v_past_Na=output_4[:,i - length(fading_weight_Na) + 1:i]
        end
        v_past=v_past_Na[:,1+(num_points_Na-num_points_K):num_points_Na]
    end


    result_Na=ion_Na(v_past_Na.-V_Na)
    #result_Na=sys4(x[2:3],[x[1]])
    result_K=sys3(x[4],[x[1]])
    
    # Update cell membrane voltage, Vm
    dVm = (input - result_Na[1] - result_K[2][1]  - (x[1]-V_L)*G_L)/C_m 

    x[1]=x[1]+dVm*dt

    # if i>=1000
    #     output_3[1,i+1]=x[1]
    # else
    #     output_3[1,i+1]=output_o_re[1,i+1]
    #     x[1]=output_o_re[1,i+1]
    # end

    output_4[1,i+1]=x[1]
    
    x[4]=x[4]+result_K[1]*dt
    output_K3[1,i+1]=result_K[2][1]
    output_Na3[1,i+1]=result_Na[1]
end

plot_Na=plot([output_4' output_K3' output_Na3'])
savefig(plot_Na,"HH_model_app_Na_test.png")
plot_Na

In [ ]:
plot([output_o output_4'])

In [ ]:
output_5=ones(1,4000)*(-65.1)
output_K4=zeros(1,4000)
output_Na4=zeros(1,4000)
rest_potential=-65.1
#x=[-65.1,m_∞(0),h_∞(0),n_∞(0)] #v,m,h,n

x=[-65.1,0,0] #v,m,h,n


dt=0.01

for i=1:3999

    if num_points_Na<num_points_K
        if i<num_points_K
            v_past=[ones(1,num_points_K - i)*rest_potential copy(output_5[:,1:i])]
        else
            v_past=output_5[:,i - length(fading_weight_K) + 1:i]
        end
        v_past_Na=v_past[:,1+(num_points_K-num_points_Na):num_points_K]
    else
        if i<num_points_Na
            v_past_Na=[ones(1,num_points_Na - i)*rest_potential copy(output_5[:,1:i])]
        else
            v_past_Na=output_5[:,i - length(fading_weight_Na) + 1:i]
        end
        v_past=v_past_Na[:,1+(num_points_Na-num_points_K):num_points_Na]
    end


    result_Na=ion_Na(v_past_Na.-V_Na)
    #result_Na=sys4(x[2:3],[x[1]])
    result_K=ion_K(v_past.-V_K)
    
    # Update cell membrane voltage, Vm
    dVm = (input[i] - result_Na[1] - result_K[1]  - (x[1]-V_L)*G_L)/C_m 

    x[1]=x[1]+dVm*dt

    # if i>=1000
    #     output_5[1,i+1]=x[1]
    # else
    #     output_5[1,i+1]=output_o_re[1,i+1]
    #     x[1]=output_o_re[1,i+1]
    # end

    output_5[1,i+1]=x[1]
    

    output_K4[1,i+1]=result_K[1]
    output_Na4[1,i+1]=result_Na[1]
end

plot_Na_K=plot([output_5' output_K4' output_Na4'])
savefig(plot_Na_K,"HH_model_appK_app_Na_test2.png")
plot_Na_K

In [ ]:
plot_Na_K=plot([output_5' output_K4' output_Na4'])

In [ ]:
output_noise=[-65.1]
output_K_noise=[0.0]
output_Na_noise=[0.0]
x=[-65.1,m_∞(-65.1),h_∞(-65.1),n_∞(-65.1)] #v,m,h,n

#x=[-65.1,0.0,0.0,0.0] #v,m,h,n
dt=0.01

for i=1:3999
      
    result_Na=sys4(x[2:3],[x[1]])
    result_K=sys3(x[4],[x[1]])
    
    # Update cell membrane voltage, Vm
    dVm = (input[i] -result_Na[2][1] - result_K[2][1]  - (x[1]-V_L)*G_L)/C_m 
    
    push!(output_noise,x[1])
    x[1]=x[1]+dVm*dt
    x[2:3]=x[2:3]+result_Na[1]*dt
    x[4]=x[4]+result_K[1]*dt
   
    push!(output_K_noise,result_K[2][1])
    push!(output_Na_noise,result_Na[2][1])
end

plot_noise=plot([output_noise output_K_noise output_Na_noise])

In [ ]:
plot(input)

In [ ]:

t_range = LinRange(0, 40, 4000)
plot_test2_summary=plot(t_range,[output_noise output_5'],label=["original" "approx_K_Na" ],xlabel="time/s", ylabel="mV")
plot_test2_input=plot(t_range,input,xlabel="time/s", ylabel="Current/mA")

savefig(plot_test2_summary,"HH_model_test2_summary.png")
savefig(plot_test2_input,"HH_model_test2_summary_input.png")

writedlm("test2_input.txt", input)

plot_test2_summary


In [ ]:
noise=rand(4500)*100 .-50
input=zeros(4000)
for i=501:4500
    input[i-500]=sum(noise[i-500:i])/100 +3
end
plot(input)

In [ ]:
num_outputs,input_size,output_size,num_points=(1000,1,1,2000)
num_outputs_temp=50
num_outputs_temp1=30
num_outputs_temp2=20
amplification=1
lambda=0.2
lambda2=0
time_range=20
dt=time_range/num_points
println(dt)
sigma=1
num_point_model=1000
fading_weight_time=10
x=Array{Float64,3}(undef,(num_outputs,input_size,num_point_model))
y=Array{Float64,3}(undef,(num_outputs,output_size,num_point_model))
x_temp=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
t=range(0,time_range,length=num_points)


t_fading=range(0,fading_weight_time,num_point_model)
fading_weight=Array([exp(lambda*(t_fading[i]-fading_weight_time)) for i=1:num_point_model]');
fading_weight2=Array([exp(lambda2*(t_fading[i]-fading_weight_time)) for i=1:num_point_model]');
#fading_weight[951:1000]=zeros(50)
fading_weight=fading_weight/(sum(fading_weight.^2)*dt)^0.5*amplification;
fading_weight2=fading_weight2/(sum(fading_weight2.^2)*dt)^0.5*amplification;


# Threads.@threads for i=0:4
#     amp=(i+1)*0.2
#     tsample,x_temp[1+i*4:4+i*4,:,:],y_temp[1+i*4:4+i*4,:,:]=Signal_output_sin(sys,2,x0,[20],[amp],num_points,input_size,output_size,(num_outputs-20)÷50)
# end

# x[1:20,:,:]=x_temp
# y[1:20,:,:]=y_temp

# Threads.@threads for i=2:19    
#     x[1+i*10:10+i*10,:,:]=slice_fading_input(x_temp[i-1,:,:], fading_weight,1/100)
#     y[1+i*10:10+i*10,:,:]=slice_fading_input(y_temp[i-1,:,:], fading_weight,1/100)
# end
# step=Int(num_outputs/100)
# for i=0:99
#     amp=(i+1)*0.2
#     tsample,x_temp[1+i*step:step+i*step,:,:],y_temp[1+i*step:step+i*step,:,:]=Signal_output_sin(sys,2,x0,[10],[amp],[i*pi/5],num_points,input_size,output_size,(num_outputs)÷100,lsim)
# end

# x=copy(x_temp)
# y=copy(y_temp)

# for i=1:num_outputs_temp
#     bias=-0.1*i+1
#     tsample,x_temp[i,:,:],y_temp[i,:,:]=Signal_output_step(sys3,[[10,time_range*1.5,time_range]], [0], [-1],[100],[bias],num_points,input_size,output_size,1,nlsim)
# end

for i=1:num_outputs_temp
    bias=-2*i+30+65.1
    # tsample,x_temp[1+(i-1)*50:i*50,:,:],y_temp[1+(i-1)*50:i*50,:,:]=Signal_output_sin(sys3,time_range, [0], [1], [1],[0], [bias], num_points,input_size,output_size,50,nlsim)
    tsample,x_temp[i,:,:],y_temp[i,:,:]=Signal_output_step(sys3,
        [[10,time_range*1.5,time_range]], [n_∞(-65.1)], [bias], [1000], [-65.1], 
        num_points,input_size,output_size,1,nlsim)
    # tsample,x_temp[i,:,:],y_temp[i,:,:]=Signal_output_sin(sys3,time_range, [n_∞(-65.1)], [0], [0],[0], [bias], num_points,input_size,output_size,1,nlsim)
end


x_temp=x_temp.-V_K

ratio=1/50
start_position=Int(1000)
step=Int((num_points-start_position)*ratio)
Threads.@threads for i=1:num_outputs_temp    
    x[1+(i-1)*step:step+(i-1)*step,:,:]=slice_fading_input(x_temp[i,:,:], ones(size(fading_weight)),ratio,-65.1-V_K,start_position)
    y[1+(i-1)*step:step+(i-1)*step,:,:]=slice_fading_input(y_temp[i,:,:], ones(size(fading_weight)),ratio,0.0,start_position)
end

# x[num_outputs_temp*step+1:num_outputs_temp*step+80,:,:]=slice_fading_input(output_o_re.-V_K, ones(size(fading_weight)),ratio,-65.1-V_K)
# y[num_outputs_temp*step+1:num_outputs_temp*step+80,:,:]=slice_fading_input(output_K_o_re, ones(size(fading_weight)),ratio)

# x=copy(x_temp)
# y=copy(y_temp)

x_train=Array{Float64,3}(undef,(num_outputs,input_size,num_point_model))
for i in range(1,size(x)[1])
    x_train[i,:,:]=x[i,:,:].*fading_weight
end

y_train=y[:,1,size(y)[3]]


# rbf=rbf_kernel_signal_gen(sigma,dt)

alpha=1
beta=1
# cone=cone_kernel_signal_gen(alpha,beta,dt)
# exp_kernel=exp_kernel_signal_gen(1,dt)
Sigma=10
Sigma2=10

gain_schedule_kernel=gain_schedule_kernel_gen4_v3_7(Sigma,dt,num_point_model)
kernel_K=gain_schedule_kernel_gen4_v3_7(Sigma,dt,num_point_model)


# gain_schedule_kernel=gain_schedule_kernel_gen(Sigma,dt)
# kernel_K=gain_schedule_kernel_gen(Sigma,dt)

# gain_schedule_kernel=gain_schedule_kernel_max_gen(alpha,dt)
# kernel_K=gain_schedule_kernel_max_gen(alpha,dt)


K_gain=kernel_matrix(x_train,gain_schedule_kernel)
#K=kernel_matrix(x_train,rbf)
#K_cone=kernel_matrix(x_train,cone)
# K_exp=kernel_matrix(x_train,exp_kernel)
# K_sigmoid=kernel_matrix(x_train,sigmoid)

weight_gain=ridge_regression_kernel(K_gain, y_train, 0.0001)
#weight=ridge_regression_kernel(K, y_train, 0.003)
#weight_cone=ridge_regression_kernel(K_cone, y_train, 0.000001)
# weight_exp=ridge_regression_kernel(K_exp,y_train,0.8)
# weight_sigmoid=ridge_regression_kernel(K_sigmoid,y_train,0.1)
# println(maximum(weight_exp))
# println(maximum(K_exp))
# println(maximum(weight_sigmoid))

error_gain=sum(abs.(y_train-K_gain*weight_gain))/size(y)[1]
#error=sum(abs.(y_train-K*weight))/size(y)[1]
#error_cone=sum(abs.(y_train-K_cone*weight_cone))/size(y)[1]
# error_exp=sum(abs.(y_train-K_exp*weight_exp))/size(y)[1]
# error_sigmoid=sum(abs.(y_train-K_sigmoid*weight_sigmoid))/size(y)[1]

println(error_gain)
println((weight_gain'*K_gain*weight_gain)^0.5*(sum(fading_weight.^2)*dt)^0.5)

x_K=copy(x)
y_K=copy(y)
x_train_K=copy(x_train)
y_train_K=copy(y_train)
weight_K=copy(weight_gain)
fading_weight_K=copy(fading_weight)
time_range_K=copy(time_range);
num_outputs_K=copy(num_outputs)
input_size_K=copy(input_size)
num_points_K=copy(num_point_model)

function ion_K(input)
    input_fade=Array{Float64,3}(undef,(1,input_size_K,num_points_K))
    input_fade[1,:,:]=(input).*fading_weight_K
    K_star=kernel_matrix(input_fade,x_train_K,kernel_K)
    return K_star*weight_K
end

#println(error)
#println(error_cone)
#println((weight'*K*weight)^0.5*(sum(fading_weight.^2)*dt)^0.5)
#println(alpha*sum(abs.(weight_cone))*(sum(fading_weight.^2)*dt)^0.5)
# println(error_exp)
# println(error_sigmoid)

In [ ]:
output_K_re=copy(reshape(output_K,(1,4000)))
output_2_re=copy(reshape(output,(1,4000)))
output_o_re=copy(reshape(output_o,(1,4000)))
output_K_o_re=copy(reshape(output_K_o,(1,4000)))



x_demo=slice_fading_input(output_o_re.-V_K, fading_weight_K,1,-65.1-V_K)


K_star=kernel_matrix(x_demo,x_train_K,kernel_K)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_K;


#y_demo=K_star_sigmoid*weight_sigmoid;




In [ ]:
plot_K_open_loop=plot([y_demo output_K_o_re'],label=[ "approx_K" "original"],xlabel="time steps", ylabel="current output")
savefig(plot_K_open_loop,"ion_channel_K_open_loop.png")
plot_K_open_loop

In [ ]:
support_vector_inte=reshape(sum(x_train_K,dims=(2,3)),size(x_train_K)[1])
support_vector_l2=reshape(sum(x_train_K.^2,dims=(2,3)).*dt,size(x_train_K)[1]).^0.5
support_vector_normal_inte=support_vector_inte./support_vector_l2
support_vector_first_ele_norm=reshape(sum(abs.(x_train_K[:,:,num_points_K]),dims=2),size(x_train_K)[1])
gradient_kernel_part1(x)=1/(2*Sigma^2)*sum(exp.(-(support_vector_inte-support_vector_normal_inte*x).^2 ./(2*Sigma^2))*2 .*(  support_vector_inte.-support_vector_normal_inte.*x).*support_vector_l2.*support_vector_first_ele_norm.*x^2)
gradient_kernel_part1_neg(x)=-1/(2*Sigma^2)*sum(exp.(-(support_vector_inte-support_vector_normal_inte*x).^2 ./(2*Sigma^2))*2 .*(  support_vector_inte.-support_vector_normal_inte.*x).*support_vector_l2.*support_vector_first_ele_norm.*x^2)

res_min=optimize( gradient_kernel_part1, -maximum(abs.(support_vector_inte))*10, maximum(abs.(support_vector_inte)*10))
res_max=optimize( gradient_kernel_part1_neg, -maximum(abs.(support_vector_inte))*10, maximum(abs.(support_vector_inte)*10))

maximum_gradient=-minimum(res_max)
minimum_gradient=minimum(res_min)

gradient_kernel_part2(y,x)=exp(-(y-x)^2/(2*Sigma^2))
A=[[zeros(num_points-1)'; I] zeros(num_points_K)]
B=[1 ;zeros(num_points_K-1)]
x = Variable(size(x_train_K)[1])
P = Variable(size(x_train_K)[3],size(x_train_K)[3])
R = Variable(size(x_train_K)[3],size(x_train_K)[3])
problem = minimize(sumsquares(y_train-K_gain*x))
for i=1:size(x_train_K)[1]
    C=x[i]*x_train_K[i,1,num_points_K]*(abs(support_vector_inte[i])+Sigma)*(reshape(x_train_K[i,1,:],num_points_K)*dt)
    C2=-x[i]*x_train_K[i,1,num_points_K]*(abs(support_vector_inte[i])+Sigma)*(reshape(x_train_K[i,1,:],num_points_K)*dt)
    C3=x[i]*support_vector_l2[i]*(abs(support_vector_inte[i])+Sigma)*[zeros(num_points_K-1);x_train_K[i,1,num_points_K]]
    C4=-x[i]*support_vector_l2[i]*(abs(support_vector_inte[i])+Sigma)*[zeros(num_points_K-1);x_train_K[i,1,num_points_K]]
   

    problem.constraints+=([A'*P*A-P+R A'*P*B-0.5*C; B'*P*A-0.5*C' B'*P*B]<0)
    problem.constraints+=([A'*P*A-P+R A'*P*B-0.5*C2; B'*P*A-0.5*C2' B'*P*B]<0)
    problem.constraints+=([A'*P*A-P+R A'*P*B-0.5*C3; B'*P*A-0.5*C3' B'*P*B]<0)
    problem.constraints+=([A'*P*A-P+R A'*P*B-0.5*C4; B'*P*A-0.5*C4' B'*P*B]<0)
end
# C5=maximum_gradient*fading_weight_K
# C6=-maximum_gradient*fading_weight_K
# problem.constraints+=([A'*P*A-P+R A'*P*B-0.5*C5'; B'*P*A-0.5*C5 B'*P*B]<0)
# problem.constraints+=([A'*P*A-P+R A'*P*B-0.5*C6'; B'*P*A-0.5*C6 B'*P*B]<0)

Convex.solve!(problem, SCS.Optimizer; silent_solver = true)

In [ ]:
support_vector_inte=reshape(sum(x_train_K,dims=(2,3)).*dt,size(x_train_K)[1])
support_vector_l2=reshape(sum(x_train_K.^2,dims=(2,3)).*dt,size(x_train_K)[1]).^0.5
support_vector_normal_inte=support_vector_inte./support_vector_l2
support_vector_first_ele_norm=reshape(sum(abs.(x_train_K[:,:,num_points_K]),dims=2),size(x_train_K)[1])
gradient_kernel_part1(x)=1/(2*Sigma^2)*sum(exp.(-(support_vector_inte-support_vector_normal_inte*x).^2 ./(2*Sigma^2))*2 .*(  support_vector_inte.-support_vector_normal_inte.*x).*support_vector_l2.*support_vector_first_ele_norm.*x^2)

gradient_kernel_part2(y,x)=exp(-(y-x)^2/(2*Sigma^2))

gradient_kernel_part1_neg(x)=-1/(2*Sigma^2)*sum(exp.(-(support_vector_inte-support_vector_normal_inte*x).^2 ./(2*Sigma^2))*2 .*(  support_vector_inte.-support_vector_normal_inte.*x).*support_vector_l2.*support_vector_first_ele_norm.*x^2)

res_min=optimize( gradient_kernel_part1, -maximum(abs.(support_vector_inte))*10, maximum(abs.(support_vector_inte)*10))
res_max=optimize( gradient_kernel_part1_neg, -maximum(abs.(support_vector_inte))*10, maximum(abs.(support_vector_inte)*10))

maximum_gradient=-minimum(res_max)
minimum_gradient=minimum(res_min)

print(maximum_gradient,"   ",minimum_gradient)


In [ ]:
support_vector_inte=reshape(sum(x_train_K,dims=(2,3)).*dt,size(x_train_K)[1])
gradient_kernel_part1_vec(x)=-1/(2*Sigma^2)*sum(exp.(-(support_vector_inte.-sum(x)*dt).^2 ./(2*Sigma^2))*2 .*(  support_vector_inte.-sum(x)*dt).*sum(x_train_K.*x,dims=(2,3))*dt.*x_train_K[:,1,100].*x[100])
lower = ones(100)*-10000
upper = ones(100)*10000
x0=rand(100)*10000
optimize( gradient_kernel_part1_vec,lower,upper,x0)

In [ ]:
num_point_model=1000
fading_weight_time=10
t_fading=range(0,fading_weight_time,num_point_model)

x_scatter_temp,y_scatter_temp=scattering_trans(x_temp,y_temp)
x_scatter=Array{Float64,3}(undef,(num_outputs,input_size,num_point_model))
y_scatter=Array{Float64,3}(undef,(num_outputs,output_size,num_point_model))

amplification=1
lambda_scatter=1
fading_weight_scatter=Array([exp(lambda_scatter*(t_fading[i]-fading_weight_time)) for i=1:length(t_fading)]')

fading_weight_scatter=fading_weight_scatter/(sum(fading_weight_scatter.^2)*dt)^0.5*amplification;


ratio=1/50
start_position=Int(1000)
step=Int((num_points-start_position)*ratio)
Threads.@threads for i=1:num_outputs_temp    
    x_scatter[1+(i-1)*step:step+(i-1)*step,:,:]=slice_fading_input(x_scatter_temp[i,:,:], ones(size(fading_weight_scatter)),ratio,-65.1-V_K,start_position)
    y_scatter[1+(i-1)*step:step+(i-1)*step,:,:]=slice_fading_input(y_scatter_temp[i,:,:], ones(size(fading_weight_scatter)),ratio,0.0,start_position)
end

x_train_scatter=Array{Float64,3}(undef,(num_outputs,input_size,num_point_model))
Threads.@threads for i in range(1,size(x)[1])
    x_train_scatter[i,:,:]=x_scatter[i,:,:].*fading_weight_scatter
end

y_train_scatter=y_scatter[:,1,size(y_scatter)[3]]

rbf=rbf_kernel_signal_gen(sigma,dt)
# beta=0.001
# alpha=0.004

Sigma=20
#gain_schedule_kernel=gain_schedule_kernel_gen7(Sigma,dt,num_points_K,1)
gain_schedule_kernel=gain_schedule_kernel_gen4_v3_8(Sigma,dt,num_point_model,0.5)



K=kernel_matrix(x_train_scatter,gain_schedule_kernel)

weight_scatter=ridge_regression_kernel(K, y_train_scatter,0.0001)



error=sum(abs.(y_train_scatter-K*weight_scatter))/size(y)[1]

println(error)
#print((sum((weight_scatter.*x_train_scatter).^2)*dt)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5)
println(((weight_scatter'*K*weight_scatter)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5))

In [ ]:
x_demo=slice_fading_input(x_scatter_demo[10,:,:], fading_weight_scatter,1,-65.1-V_K,start_position)


K_star=kernel_matrix(x_demo,x_train_scatter,gain_schedule_kernel)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_scatter;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot([y_demo[1:1000] y_scatter_demo[10,:,1001:2000]' ])

In [ ]:

tsample,x_test_,y_test_=Signal_output_sin(sys3,time_range_K, [0], [3], [3], [0],[0], num_points,input_size,output_size,10,nlsim)


y_test=y_test_[:,1,size(y_test_)[3]]
x_test_=x_test_.-V_K

x_test=zeros(size(x_test_))
for i in range(1,size(x_test_)[1])
    x_test[i,:,:]=x_test_[i,:,:].*fading_weight_K
end

K_star=kernel_matrix(x_test,x_train_K,kernel_K)
error=sum(abs.(y_test-K_star*weight_K))/size(y_test)[1]



In [ ]:
num=5
x_demo=slice_fading_input(x_test_[num,:,:], fading_weight_K,1)


K_star=kernel_matrix(x_demo,x_train_K,kernel_K)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_K;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot([y_demo[1:1000] y_test_[num,:,1:1000]' ])

In [ ]:
num=100
x_demo=slice_fading_input(x_K[num,:,:], fading_weight_K,1)


K_star=kernel_matrix(x_demo,x_train_K,kernel_K)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_K;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot([y_demo[1:1000] y_K[num,:,1:1000]' x_K[num,:,:]'])

In [ ]:
num_outputs,input_size,output_size,num_points=(2000,1,1,1500)
num_outputs_temp=40
lambda=0.2
lambda2=1.5
time_range=15
amplification=1
dt=time_range/num_points
println(dt)
sigma=1
num_point_model=1500
fading_weight_time=15
x2=Array{Float64,3}(undef,(num_outputs,input_size,num_point_model))
y2=Array{Float64,3}(undef,(num_outputs,output_size,num_point_model))
x_temp2=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp2=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
t=range(0,time_range,length=num_points)


t_fading=range(0,fading_weight_time,num_point_model)
fading_weight=Array([exp(lambda*(t_fading[i]-fading_weight_time)) for i=1:num_point_model]');
fading_weight2=Array([exp(lambda2*(t_fading[i]-fading_weight_time)) for i=1:num_point_model]');
#fading_weight[951:1000]=zeros(50)
fading_weight=fading_weight/(sum(fading_weight.^2)*dt)^0.5*amplification;
fading_weight2=fading_weight2/(sum(fading_weight2.^2)*dt)^0.5*amplification;

# output_o_re=copy(reshape(output_o,(1,4000)))
# output_Na_o_re=copy(reshape(output_Na_o,(1,4000)))


# for i=1:20
#     bias=-5*i+20
#     for j=1:5
#         step_size=i*10
#         tsample,x_temp2[(i-1)*5+j,:,:],y_temp2[(i-1)*5+j,:,:]=Signal_output_step(sys4,[[5,1.5*time_range,time_range]], [m_∞(V_Na),h_∞(V_Na)], [step_size], [70], [bias], num_points,input_size,output_size,1,nlsim)
#     end
# end

# for i=1:50
#     bias=-2*i+30
#     # tsample,x_temp[1+(i-1)*50:i*50,:,:],y_temp[1+(i-1)*50:i*50,:,:]=Signal_output_sin(sys3,time_range, [0], [1], [1],[0], [bias], num_points,input_size,output_size,50,nlsim)
#     tsample,x_temp2[i,:,:],y_temp2[i,:,:]=Signal_output_sin(sys4,time_range, [m_∞(V_Na),h_∞(V_Na)], [0], [0],[0], [bias], num_points,input_size,output_size,1,nlsim)
# end

sharpness=100
start1=13
end1=13.5
start2=13.8
end2=13.9

# for i=1:40
#     bias=-2.5*i+20

#     # u(x,t) =[1 / (exp(-sharpness * (t - start1)) + 1) * 50 * 
#     #             1 / (exp(-sharpness * (-t + end1)) + 1)+bias-
#     #             1 / (exp(-sharpness * (t - start2)) + 1) * 50 * 
#     #             1 / (exp(-sharpness * (-t + end2)) + 1)
#     #             ]
#     # u(x,t)=[bias]
#     # tsample,x_temp2[i,:,:],y_temp2[i,:,:]=Signal_output(sys4,
#     # time_range, [m_∞(-65.1),h_∞(-65.1)], u,
#     # num_points,input_size,output_size,nlsim)
#     tsample,x_temp2[i,:,:],y_temp2[i,:,:]=Signal_output_sin(sys4,time_range, [m_∞(-65.1),h_∞(-65.1)], [0], [0],[0], [bias], num_points,input_size,output_size,1,nlsim)
# end

for i=0:39
    bias=-2.5*i+20
        tsample,x_temp2[i+1,:,:],y_temp2[i+1,:,:]=Signal_output_step(sys4,
        [[13,13.01,time_range]], [m_∞(-65.1),h_∞(-65.1)], [1000], [100], [bias], 
        num_points,input_size,output_size,1,nlsim)
end

# x_temp2[41,:,:]=output_o_re[1,1:1000]
# y_temp2[41,:,:]=output_Na_o_re[1,1:1000]

# x_temp2[16,:,:]=output_o_re[1,1:1000]
# y_temp2[16,:,:]=output_Na_o_re[1,1:1000]
# x_temp2[17,:,:]=output_o_re[1,1001:2000]
# y_temp2[17,:,:]=output_Na_o_re[1,1001:2000]
# x_temp2[18,:,:]=output_o_re[1,2001:3000]
# y_temp2[18,:,:]=output_Na_o_re[1,2001:3000]
# x_temp2[19,:,:]=output_o_re[1,3001:4000]
# y_temp2[19,:,:]=output_Na_o_re[1,3001:4000]


x_temp2=x_temp2 .- V_Na

ratio=1/60
start_position=Int(0)
step=Int((num_points-start_position)*ratio)
Threads.@threads for i=1:num_outputs_temp 
    x2[1+(i-1)*step:step+(i-1)*step,:,:]=slice_fading_input(x_temp2[i,:,:], ones(size(fading_weight)),ratio,-65.1-V_Na,start_position)
    y2[1+(i-1)*step:step+(i-1)*step,:,:]=slice_fading_input(y_temp2[i,:,:], ones(size(fading_weight)),ratio,0.0,start_position)
end

ratio=1/8
start_position=Int(1300)
step=Int((num_points-start_position)*ratio)
Threads.@threads for i=1:num_outputs_temp 
    x2[1001+(i-1)*step:1000+step+(i-1)*step,:,:]=slice_fading_input(x_temp2[i,:,:], ones(size(fading_weight)),ratio,-65.1-V_Na,start_position)
    y2[1001+(i-1)*step:1000+step+(i-1)*step,:,:]=slice_fading_input(y_temp2[i,:,:], ones(size(fading_weight)),ratio,0.0,start_position)
end

# ratio=1/600
# start_position=Int(0)
# step=Int((num_points-start_position)*ratio)
# Threads.@threads for i=1:num_outputs_temp 
#     x2[2001+(i-1)*step:2000+step+(i-1)*step,:,:]=slice_fading_input(x_temp2[i,:,:], ones(size(fading_weight)),ratio,-65.1-V_Na)
#     y2[2001+(i-1)*step:2000+step+(i-1)*step,:,:]=slice_fading_input(y_temp2[i,:,:], ones(size(fading_weight)),ratio,0.0)
# end
# x2[num_outputs_temp*step+1:num_outputs_temp*step+200,:,:]=slice_fading_input(output_o_re.-V_Na, ones(size(fading_weight)),1/20,-65.1-V_Na)
# y2[num_outputs_temp*step+1:num_outputs_temp*step+200,:,:]=slice_fading_input(output_Na_o_re, ones(size(fading_weight)),1/20)

#x2[num_outputs_temp*step+201:num_outputs,:,:]=slice_fading_input(output_2_re.-V_Na, ones(size(fading_weight)),ratio,-65.1-V_Na)
#y2[num_outputs_temp*step+201:num_outputs,:,:]=slice_fading_input(output_Na_re, ones(size(fading_weight)),ratio)

x_train2=Array{Float64,3}(undef,(num_outputs,input_size,num_point_model))
for i in range(1,size(x2)[1])
    x_train2[i,:,:]=x2[i,:,:].*fading_weight
end

y_train2=y2[:,1,size(y2)[3]]


#rbf=rbf_kernel_signal_gen(sigma,dt)

#alpha=0.000001
#beta=0.1
#cone=cone_kernel_signal_gen(alpha,beta,dt)
#exp_kernel=exp_kernel_signal_gen(1,dt)
#gain_schedule_kernel=gain_schedule_kernel_gen(Sigma,dt)
alpha=1
beta=0
# cone=cone_kernel_signal_gen(alpha,beta,dt)
# exp_kernel=exp_kernel_signal_gen(1,dt)
Sigma=9.5
Sigma2=2.5
Sigma3=1
# gain_schedule_kernel=gain_schedule_kernel_gen4_v6(Sigma,dt,num_points,alpha,beta)
# kernel_Na=gain_schedule_kernel_gen4_v6(Sigma,dt,num_points,alpha,beta)

gain_schedule_kernel=gain_schedule_kernel_gen4_v3_2(Sigma,Sigma2,dt,num_point_model,fading_weight2)
kernel_Na=gain_schedule_kernel_gen4_v3_2(Sigma,Sigma2,dt,num_point_model,fading_weight2)

# gain_schedule_kernel=gain_schedule_kernel_gen4_v3_4(Sigma2,dt,num_point_model,ones(size(fading_weight)))
# kernel_Na=gain_schedule_kernel_gen4_v3_4(Sigma2,dt,num_point_model,ones(size(fading_weight)))


# gain_schedule_kernel=gain_schedule_kernel_gen4_v7(Sigma,Sigma2,Sigma3,dt,num_point_model,alpha,beta)
# kernel_Na=gain_schedule_kernel_gen4_v7(Sigma,Sigma2,Sigma3,dt,num_point_model,alpha,beta)

# gain_schedule_kernel=gain_schedule_kernel_gen4_v10(Sigma,dt,900,alpha,beta)
# kernel_Na=gain_schedule_kernel_gen4_v10(Sigma,dt,900,alpha,beta)

# gain_schedule_kernel=gain_schedule_kernel_gen4_v7(Sigma,Sigma2,Sigma3,dt,1000,alpha,beta,0.000)
# kernel_Na=gain_schedule_kernel_gen4_v7(Sigma,Sigma2,Sigma3,dt,1000,alpha,beta,0.000)

# gain_schedule_kernel=gain_schedule_kernel_gen5(Sigma,dt,num_points)
# kernel_Na=gain_schedule_kernel_gen5(Sigma,dt,num_points)

K_gain2=kernel_matrix(x_train2,gain_schedule_kernel)

#K_rbf2=kernel_matrix(x_train2,rbf)

weight_gain2=ridge_regression_kernel(K_gain2, y_train2, 0.0000001)



#residual=y_train2-K_gain2*weight_gain2

#weight_rbf2=ridge_regression_kernel(K_rbf2, residual, 0.001)

#error_gain2=sum(abs.(y_train2-K_gain2*weight_gain2-K_rbf2*weight_rbf2))/size(y2)[1]

error=sum(abs.(y_train2-K_gain2*weight_gain2))/size(y2)[1]

println(error)
#println(error_gain2)
println((weight_gain2'*K_gain2*weight_gain2)^0.5*(sum(fading_weight.^2)*dt)^0.5)

time_range_Na=copy(time_range)
fading_weight_Na=copy(fading_weight)
x_Na=copy(x2)
y_Na=copy(y2)
x_train_Na=copy(x_train2)
y_train_Na=copy(y_train2)
weight_Na=copy(weight_gain2);
num_outputs_Na=copy(num_outputs)
input_size_Na=copy(input_size)
num_points_Na=copy(num_point_model)

function ion_Na(input)
    input_fade=Array{Float64,3}(undef,(1,input_size_Na,num_points_Na))
    input_fade[1,:,:]=(input).*fading_weight_Na
    K_star=kernel_matrix(input_fade,x_train_Na,kernel_Na)
    return K_star*weight_Na
end


In [ ]:
num=20
x_demo=slice_fading_input(x_temp2[num,:,:], fading_weight_Na,1,-65.1-V_Na)


K_star=kernel_matrix(x_demo,x_train_Na,kernel_Na)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_Na;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot([y_demo y_temp2[num,:,:]'])

In [ ]:
plot([y_demo y_temp2[num,:,:]'])

In [ ]:
output_Na_re=copy(reshape(output_Na,(1,4000)))
output_2_re=copy(reshape(output,(1,4000)))
output_o_re=copy(reshape(output_o,(1,4000)))
output_Na_o_re=copy(reshape(output_Na_o,(1,4000)))



x_demo=slice_fading_input(output_o_re.-V_Na, fading_weight_Na,1,-65.1-V_Na)


K_star=kernel_matrix(x_demo,x_train_Na,kernel_Na)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_Na;


#y_demo=K_star_sigmoid*weight_sigmoid;


# plot([y_demo output_Na_re'])

In [ ]:
plot_Na_open_loop=plot([y_demo output_Na_o_re'],label=[ "approx_Na" "original"],xlabel="time steps", ylabel="current output")
savefig(plot_Na_open_loop,"ion_channel_Na_open_loop.png")
plot_Na_open_loop

In [ ]:
num_outputs,input_size,output_size,num_points=(1000,1,1,1000)
num_outputs_temp=20
lambda=5
time_range=4
dt=time_range/num_points
sigma=1
amplification=1
x=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
y=Array{Float64,3}(undef,(num_outputs,output_size,num_points))
x_temp=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_temp2=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp2=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_test=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_test=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_test2=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_test2=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
t=range(0,time_range,length=num_points)
fading_weight=Array([exp(lambda*(t[i]-time_range)) for i=1:length(t)]');
fading_weight=fading_weight/(sum(fading_weight.^2)*dt)^0.5*amplification;


step=Int(num_outputs_temp/10)
Threads.@threads for i=0:9
    amp=(i+1)
    tsample,x_test2[1+i*step:step+i*step,:,:],y_test2[1+i*step:step+i*step,:,:]=Signal_output_sin(sys,time_range,zeros(1),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
    tsample,x_temp2[1+i*step:step+i*step,:,:],y_temp2[1+i*step:step+i*step,:,:]=Signal_output_sin(sys,time_range,zeros(1),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
end

# step=Int(num_outputs_temp/10)
# Threads.@threads for i=0:9
#     amp=(i+1)*1
#     tsample,x_test[1+i*step:step+i*step,:,:],y_test[1+i*step:step+i*step,:,:]=Signal_output_sin(sys,time_range,zeros(1),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
# end

for i=1:num_outputs_temp
    bias=1*i-10
    # tsample,x_temp[1+(i-1)*50:i*50,:,:],y_temp[1+(i-1)*50:i*50,:,:]=Signal_output_sin(sys3,time_range, [0], [1], [1],[0], [bias], num_points,input_size,output_size,50,nlsim)
    tsample,x_temp[i,:,:],y_temp[i,:,:]=Signal_output_sin(sys,time_range, zeros(1), [0], [0],[0], [bias], num_points,input_size,output_size,1)
    tsample,x_test[i,:,:],y_test[i,:,:]=Signal_output_sin(sys,time_range, zeros(1), [0], [0],[0], [bias], num_points,input_size,output_size,1)
end



ratio=1/20
step=Int(num_points*ratio)
Threads.@threads for i=0:num_outputs_temp-1    
    x[1+i*step:step+i*step,:,:]=slice_fading_input(x_temp2[i+1,:,:], ones(size(fading_weight)),ratio)
    y[1+i*step:step+i*step,:,:]=slice_fading_input(y_temp2[i+1,:,:], ones(size(fading_weight)),ratio)
end

x_train=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
Threads.@threads for i in range(1,size(x)[1])
    x_train[i,:,:]=x[i,:,:].*fading_weight
end

y_train=y[:,1,size(y)[3]]

w=ones(size(x_train[1,:,:]))
rbf=rbf_kernel_signal_gen(sigma,dt)
rbf_w=rbf_w_kernel_signal_gen(sigma,dt,w)
gain_schedule_kernel=gain_schedule_kernel_gen(1,dt)

K=kernel_matrix(x_train,gain_schedule_kernel)
K_rbf=kernel_matrix(x_train,rbf)
# K_exp=kernel_matrix(x_train,exp_kernel)
# K_sigmoid=kernel_matrix(x_train,sigmoid)

weight=ridge_regression_kernel(K, y_train, 0.0006)
weight_rbf=ridge_regression_kernel(K_rbf, y_train, 0.0006)
# weight_exp=ridge_regression_kernel(K_exp,y_train,0.8)
# weight_sigmoid=ridge_regression_kernel(K_sigmoid,y_train,0.1)
# println(maximum(weight_exp))
# println(maximum(K_exp))
# println(maximum(weight_sigmoid))

error=sum(abs.(y_train-K*weight))/size(y)[1]
error_rbf=sum(abs.(y_train-K_rbf*weight_rbf))/size(y)[1]
# error_exp=sum(abs.(y_train-K_exp*weight_exp))/size(y)[1]
# error_sigmoid=sum(abs.(y_train-K_sigmoid*weight_sigmoid))/size(y)[1]

println(error)
println(error_rbf)
println((weight'*K*weight)^0.5*(sum(fading_weight.^2)*dt)^0.5)
# println(error_exp)
# println(error_sigmoid)

In [ ]:
num_outputs,input_size,output_size,num_points=(1000,1,1,1000)
num_outputs_temp=20
lambda=4
time_range=4
dt=time_range/num_points
sigma=1
amplification=1
x=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
y=Array{Float64,3}(undef,(num_outputs,output_size,num_points))
x_temp=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_temp2=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp2=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_test=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_test=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_test2=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_test2=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
t=range(0,time_range,length=num_points)
fading_weight=Array([exp(lambda*(t[i]-time_range)) for i=1:length(t)]');
fading_weight=fading_weight/(sum(fading_weight.^2)*dt)^0.5*amplification;


step=Int(num_outputs_temp/10)
Threads.@threads for i=0:9
    amp=(i+1)
    tsample,x_test2[1+i*step:step+i*step,:,:],y_test2[1+i*step:step+i*step,:,:]=Signal_output_sin(sys7,time_range,zeros(2),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
    tsample,x_temp2[1+i*step:step+i*step,:,:],y_temp2[1+i*step:step+i*step,:,:]=Signal_output_sin(sys7,time_range,zeros(2),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
end

# step=Int(num_outputs_temp/10)
# Threads.@threads for i=0:9
#     amp=(i+1)*1
#     tsample,x_test[1+i*step:step+i*step,:,:],y_test[1+i*step:step+i*step,:,:]=Signal_output_sin(sys,time_range,zeros(1),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
# end

for i=1:num_outputs_temp
    bias=1*i-10
    # tsample,x_temp[1+(i-1)*50:i*50,:,:],y_temp[1+(i-1)*50:i*50,:,:]=Signal_output_sin(sys3,time_range, [0], [1], [1],[0], [bias], num_points,input_size,output_size,50,nlsim)
    tsample,x_temp[i,:,:],y_temp[i,:,:]=Signal_output_sin(sys7,time_range, zeros(2), [0], [0],[0], [bias], num_points,input_size,output_size,1)
    tsample,x_test[i,:,:],y_test[i,:,:]=Signal_output_sin(sys7,time_range, zeros(2), [0], [0],[0], [bias], num_points,input_size,output_size,1)
end



ratio=1/20
step=Int(num_points*ratio)
Threads.@threads for i=0:num_outputs_temp-1    
    x[1+i*step:step+i*step,:,:]=slice_fading_input(x_temp2[i+1,:,:], ones(size(fading_weight)),ratio)
    y[1+i*step:step+i*step,:,:]=slice_fading_input(y_temp2[i+1,:,:], ones(size(fading_weight)),ratio)
end

x_train=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
Threads.@threads for i in range(1,size(x)[1])
    x_train[i,:,:]=x[i,:,:].*fading_weight
end

y_train=y[:,1,size(y)[3]]

w=ones(size(x_train[1,:,:]))
rbf=rbf_kernel_signal_gen(sigma,dt)
rbf_w=rbf_w_kernel_signal_gen(sigma,dt,w)
gain_schedule_kernel=gain_schedule_kernel_gen(1,dt)

K=kernel_matrix(x_train,gain_schedule_kernel)
K_rbf=kernel_matrix(x_train,rbf)
# K_exp=kernel_matrix(x_train,exp_kernel)
# K_sigmoid=kernel_matrix(x_train,sigmoid)

weight=ridge_regression_kernel(K, y_train, 0.0006)
weight_rbf=ridge_regression_kernel(K_rbf, y_train, 0.0006)
# weight_exp=ridge_regression_kernel(K_exp,y_train,0.8)
# weight_sigmoid=ridge_regression_kernel(K_sigmoid,y_train,0.1)
# println(maximum(weight_exp))
# println(maximum(K_exp))
# println(maximum(weight_sigmoid))

error=sum(abs.(y_train-K*weight))/size(y)[1]
error_rbf=sum(abs.(y_train-K_rbf*weight_rbf))/size(y)[1]
# error_exp=sum(abs.(y_train-K_exp*weight_exp))/size(y)[1]
# error_sigmoid=sum(abs.(y_train-K_sigmoid*weight_sigmoid))/size(y)[1]

println(error)
println(error_rbf)
println((weight'*K*weight)^0.5*(sum(fading_weight.^2)*dt)^0.5)
# println(error_exp)
# println(error_sigmoid)

In [ ]:
num=18
x_demo=slice_fading_input(x_test[num,:,:], fading_weight,1)


K_star=kernel_matrix(x_demo,x_train,gain_schedule_kernel)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot_test=plot([y_demo[1:1000] y_test[num,:,1:1000]' ])
# savefig(plot_test,"stepTrainSinTest_GSK.png")
# savefig(plot_test,"sinTrainStepTest_GSK.png")
plot_test

In [ ]:
x_demo=slice_fading_input(x_test[num,:,:], fading_weight,1)


K_star=kernel_matrix(x_demo,x_train,rbf)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_rbf;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot_test=plot([y_demo[1:1000] y_test[num,:,1:1000]' ])
# savefig(plot_test,"stepTrainSinTest_RBF.png")
# savefig(plot_test,"sinTrainStepTest_RBF.png")
plot_test

In [ ]:
num_outputs,input_size,output_size,num_points=(1000,1,1,1000)
num_outputs_temp=10
lambda=5
time_range=2
dt=time_range/num_points
sigma=1
amplification=1
x=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
y=Array{Float64,3}(undef,(num_outputs,output_size,num_points))
x_temp=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_temp2=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_temp2=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_test=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_test=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
x_test2=Array{Float64,3}(undef,(num_outputs_temp,input_size,num_points))
y_test2=Array{Float64,3}(undef,(num_outputs_temp,output_size,num_points))
t=range(0,time_range,length=num_points)
fading_weight=Array([exp(lambda*(t[i]-time_range)) for i=1:length(t)]');
fading_weight=fading_weight/(sum(fading_weight.^2)*dt)^0.5*amplification;


step=Int(num_outputs_temp/10)
Threads.@threads for i=0:9
    amp=(i+1)
    tsample,x_temp[1+i*step:step+i*step,:,:],y_temp[1+i*step:step+i*step,:,:]=Signal_output_sin(sys6,time_range,zeros(1),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
end

# step=Int(num_outputs_temp/10)
# Threads.@threads for i=0:9
#     amp=(i+1)*1
#     tsample,x_test[1+i*step:step+i*step,:,:],y_test[1+i*step:step+i*step,:,:]=Signal_output_sin(sys,time_range,zeros(1),[10],[amp],[0],[0],num_points,input_size,output_size,(num_outputs_temp)÷10)
# end



ratio=1/10
step=Int(num_points*ratio)
Threads.@threads for i=0:num_outputs_temp-1    
    x[1+i*step:step+i*step,:,:]=slice_fading_input(x_temp[i+1,:,:], ones(size(fading_weight)),ratio)
    y[1+i*step:step+i*step,:,:]=slice_fading_input(y_temp[i+1,:,:], ones(size(fading_weight)),ratio)
end

x_train=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
Threads.@threads for i in range(1,size(x)[1])
    x_train[i,:,:]=x[i,:,:].*fading_weight
end

y_train=y[:,1,size(y)[3]]

w=ones(size(x_train[1,:,:]))
rbf=rbf_kernel_signal_gen(sigma,dt)
rbf_w=rbf_w_kernel_signal_gen(sigma,dt,w)
gain_schedule_kernel=gain_schedule_kernel_gen(1,dt)

K=kernel_matrix(x_train,gain_schedule_kernel)
K_rbf=kernel_matrix(x_train,rbf)
# K_exp=kernel_matrix(x_train,exp_kernel)
# K_sigmoid=kernel_matrix(x_train,sigmoid)

weight=ridge_regression_kernel(K, y_train, 0.0006)
weight_rbf=ridge_regression_kernel(K_rbf, y_train, 0.0006)
# weight_exp=ridge_regression_kernel(K_exp,y_train,0.8)
# weight_sigmoid=ridge_regression_kernel(K_sigmoid,y_train,0.1)
# println(maximum(weight_exp))
# println(maximum(K_exp))
# println(maximum(weight_sigmoid))

error=sum(abs.(y_train-K*weight))/size(y)[1]
error_rbf=sum(abs.(y_train-K_rbf*weight_rbf))/size(y)[1]
# error_exp=sum(abs.(y_train-K_exp*weight_exp))/size(y)[1]
# error_sigmoid=sum(abs.(y_train-K_sigmoid*weight_sigmoid))/size(y)[1]

println(error)
println(error_rbf)
println((weight'*K*weight)^0.5*(sum(fading_weight.^2)*dt)^0.5)
# println(error_exp)

In [ ]:
x_scatter,y_scatter=scattering_trans(x,y)

amplification=1
lambda_scatter=10
fading_weight_scatter=Array([exp(lambda_scatter*(t[i]-time_range)) for i=1:length(t)]')

fading_weight_scatter=fading_weight_scatter/(sum(fading_weight_scatter.^2)*dt)^0.5*amplification;

x_train_scatter=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
Threads.@threads for i in range(1,size(x)[1])
    x_train_scatter[i,:,:]=x_scatter[i,:,:].*fading_weight_scatter
end

y_train_scatter=y_scatter[:,1,size(y_scatter)[3]]

rbf=rbf_kernel_signal_gen(sigma,dt)
# beta=0.001
# alpha=0.004

alpha=0.0000009
beta=0.0000009
sigma=1
cone=cone_kernel_signal_gen(alpha,beta,dt)
# gain_schedule_kernel=gain_schedule_kernel_gen3(1,dt,num_points)
gain_schedule_kernel=gain_schedule_kernel_gen(10,dt)
liner=linear_kernel_signal_gen(dt)

rbf_I=rbf_I_kernel_signal_gen(sigma,dt,num_points,alpha,beta)
rbf=rbf_kernel_signal_gen(sigma,dt)


K=kernel_matrix(x_train_scatter,rbf_I)


weight_scatter=ridge_regression_kernel(K, y_train_scatter.*amplification,0.000001)



error=sum(abs.(y_train_scatter-K*weight_scatter./amplification))/size(y)[1]

println(error)
#print((sum((weight_scatter.*x_train_scatter).^2)*dt)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5)
println(((weight_scatter'*K*weight_scatter)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5)/sigma/amplification)
println((weight_scatter'*K*weight_scatter)^0.5*(beta+alpha*sum(fading_weight_scatter.^2)*dt)^0.5/sigma/amplification)
println(((weight_scatter'*K*weight_scatter)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5)*alpha*2^0.5)
println(alpha*sum(abs.(weight_scatter))*(sum(fading_weight_scatter.^2)*dt)^0.5/amplification)


In [ ]:
plot([x_scatter[100,:,:]' y_scatter[100,:,:]'])

In [ ]:
plot([x[100,:,:]',y[100,:,:]'])

In [ ]:
num=500
x_demo=slice_fading_input(x_scatter[num,:,:], fading_weight_scatter,1)


K_star=kernel_matrix(x_demo,x_train_scatter,rbf_I)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_scatter;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot([y_demo[1:1000]./amplification y_scatter[num,:,1:1000]' ])

In [ ]:
plot(x[100,:,:]'.-y_demo*2 ./amplification)

In [ ]:
plot(y[100,:,:]')

In [ ]:
num=500
sim_output=rbf_simulation_invers(x_train_scatter,weight_scatter,x[num,:,:],fading_weight_scatter,sigma,dt,100000,10^(-6),"rbf_I",alpha,beta)

final_sim_output=x[num,:,:]-2*sim_output./amplification

plot([final_sim_output'[1:1000] y[num,:,:]'[1:1000] ])

In [ ]:
x_demo=slice_fading_input(x_scatter[500,:,:], fading_weight_scatter,1)


K_star=kernel_matrix(x_demo,x_train_scatter,rbf_I)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_scatter;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot([x_scatter[500,:,:]' y_demo[1:1000] y_scatter[500,:,1:1000]' sim_output'])

In [ ]:
plot([x[50,:,:]' y_scatter[150,:,:]' x_scatter[150,:,:]'])

In [ ]:
size((x_train_scatter[:,:,length_fading_weight-i+1:length_fading_weight-1].-(x_input[:,1:i-1]-output[:,1:i-1]).*fading_weight[:,length_fading_weight-i+1:length_fading_weight-1]).^2)



In [ ]:
println(size([zeros(size(x_input)[1],length_fading_weight - i) (x_input[:,1:i-1])]))

println(size(x_train_scatter[:,:,1:length_fading_weight-1]))

In [ ]:
plot(x_train_scatter[200,:,:]')

In [ ]:
# w=x_train[1,:,:]
# for i=2:size(x_train)[1]
#     w=w+x_train[i,:,:]
# end
# w=w./size(x_train)[1]

w=ones(size(x_train[1,:,:]))

normalised_x_train=copy(x_train)

for i in 1:size(x_train)[1]
    normalised_x_train[i,:,:]=normalised_x_train[i,:,:]/(sqrt(sum((x_train[i,:,:].^2))*dt))
end

alpha=ones(size(x_train)[1])./size(x_train)[1]

step=1

minimum_e=100
minimum_w=copy(w)
error_pre=100

for i=1:5

    vibration_count=0
    trial=0

    momentum=zeros(size(x_train[1,:,:]))
    w=copy(minimum_w)

    rbf_w=rbf_w_kernel_signal_gen(sigma,dt,w)
    K=kernel_matrix(x_train,rbf_w)
    weight=ridge_regression_kernel(K, y_train, 0.01)
    y_pre=K*weight

    error_vec=copy(y_pre-y_train)
    error_conv=sum(abs.(y_train-y_pre))/size(y)[1]

    if error_conv<minimum_e
        minimum_e=copy(error_conv)
        minimum_weight=copy(weight)
    end

    if error_conv < 10^(-5)
        break
    end

    println("error_conv:",error_conv)

    for gd_iter=1:10

        gradient_dw=zeros(size(x_train[1,:,:]))

        for i2=1:size(x_train)[1]
            
            for j=1:size(x_train)[1]
                gradient_dw=gradient_dw+(weight[j]*error_vec[i2]+0.01*weight[j]*weight[i2])*rbf_w_kernel_signal_gradient_dw(sigma,dt,w,x_train[j,:,:],x_train[i2,:,:])
            end

        end    
        
        #gradient_da=[sum(x_train[i3,:,:].*gradient_dw) for i3=1:size(x_train)[1]]
        gradient_dw=copy(gradient_dw+0.0*w)

        dw=zeros(size(x_train[1,:,:]))

        for i2=1:size(x_train)[1]
            dw=dw+sum(normalised_x_train[i2,:,:].*gradient_dw*dt)*normalised_x_train[i2,:,:]
        end
        

        w=w-(0.9*momentum+dw*step)
        momentum=0.9*momentum+dw*step

        rbf_w=rbf_w_kernel_signal_gen(sigma,dt,w)
        K=kernel_matrix(x_train,rbf_w)
        y_pre=K*weight
        error=sum(abs.(y_train-y_pre))/size(y)[1]
        error_vec=y_pre-y_train

        if error<minimum_e
            minimum_e=copy(error)
            minimum_w=copy(w)
        else 
            if error>=error_pre
                vibration_count+=1
            end
        end

        error_pre=copy(error)
        println(error)

        if error < 10^(-5)
            break
        end

        if vibration_count>=3
            step=step/1.5
            vibration_count=0
            trial+=1
            momentum=zeros(size(x_train[1,:,:]))
            w=copy(minimum_w)
            println("new step:",step)
        end

        if trial>3
            break
        end

    end
end

In [ ]:
rbf_w=rbf_w_kernel_signal_gen(sigma,dt,w)
K=kernel_matrix(x_train,rbf_w)
weight_w=ridge_regression_kernel(K, y_train, 0.00001)
y_pre=K*weight_w

error_vec=y_train-y_pre
error=sum(abs.(y_train-y_pre))/size(y)[1]
println(error)

In [ ]:
plot((w.*fading_weight)')

In [ ]:
print(size((x_train[1]-x_train[2])))

In [ ]:
num_outputs,input_size,output_size,num_points=(20000,1,1,1000)
lambda=0
time_range=2
dt=time_range/num_points
sigma=0.095
x=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
y=Array{Float64,3}(undef,(num_outputs,output_size,num_points))
x_temp=Array{Float64,3}(undef,(num_outputs÷10,input_size,num_points))
y_temp=Array{Float64,3}(undef,(num_outputs÷10,output_size,num_points))
t=range(0,time_range,length=num_points)
fading_weight=Array([exp(lambda*(t[i]-time_range)) for i=1:length(t)]');


# Threads.@threads for i=0:4
#     amp=(i+1)*0.2
#     tsample,x_temp[1+i*4:4+i*4,:,:],y_temp[1+i*4:4+i*4,:,:]=Signal_output_sin(sys,2,x0,[20],[amp],num_points,input_size,output_size,(num_outputs-20)÷50)
# end

# x[1:20,:,:]=x_temp
# y[1:20,:,:]=y_temp

# Threads.@threads for i=2:19    
#     x[1+i*10:10+i*10,:,:]=slice_fading_input(x_temp[i-1,:,:], fading_weight,1/100)
#     y[1+i*10:10+i*10,:,:]=slice_fading_input(y_temp[i-1,:,:], fading_weight,1/100)
# end
step=Int(num_outputs/50)
Threads.@threads for i=0:4
    amp=(i+1)*2
    tsample,x_temp[1+i*step:step+i*step,:,:],y_temp[1+i*step:step+i*step,:,:]=Signal_output_sin(sys,2,x0,[40],[amp],[i*pi/5],num_points,input_size,output_size,(num_outputs)÷50)
end

print(size(y))

Threads.@threads for i=0:1999    
    x[1+i*10:10+i*10,:,:]=slice_fading_input(x_temp[i+1,:,:], fading_weight,1/100)
    y[1+i*10:10+i*10,:,:]=slice_fading_input(y_temp[i+1,:,:], fading_weight,1/100)
end



x_train=x


y_train=y[:,1,size(y)[3]]

basis=basis_fun2(20,2,num_points,lambda)

x_train=map2Rn(basis, x_train)

para=rbf_random_features_parameter(size(x_train)[2],2000,0.01)

fea=rbf_random_features(x_train, 1000, para)

print(fea)

weight_rf=ridge_regression_random_features(fea, y_train, 0.00000001)

print(weight_rf)

y_rf=predict_random_features_model(fea, weight_rf)

error_rf=sum(abs.(y_train-y_rf))/size(y)[1]

# weight=ridge_regression_kernel(K, y_train, 0.00001)
# # weight_exp=ridge_regression_kernel(K_exp,y_train,0.8)
# # weight_sigmoid=ridge_regression_kernel(K_sigmoid,y_train,0.1)
# # println(maximum(weight_exp))
# # println(maximum(K_exp))
# # println(maximum(weight_sigmoid))

# error=sum(abs.(y_train-K*weight))/size(y)[1]
# # error_exp=sum(abs.(y_train-K_exp*weight_exp))/size(y)[1]
# # error_sigmoid=sum(abs.(y_train-K_sigmoid*weight_sigmoid))/size(y)[1]

# println(error)
# # println(error_exp)
# # println(error_sigmoid)

In [ ]:
num_outputs_test,input_size_test,output_size_test,num_points_test=(1000,1,1,1000)
time_range_test=2
dt=time_range_test/num_points_test
x_test=Array{Float64,3}(undef,(num_outputs_test,input_size_test,num_points_test))
y_test_full=Array{Float64,3}(undef,(num_outputs_test,output_size_test,num_points_test))
t=range(0,time_range_test,length=num_points_test)


    
    
    
tsample,x_temp,y_temp=Signal_output_step(sys, [[0,1,2]], x0, [2],[100], num_points_test,input_size_test,output_size_test,div(num_outputs_test,10))

Threads.@threads for i=0:99    
    x_test[1+i*10:10+i*10,:,:]=slice_fading_input(x_temp[i+1,:,:], fading_weight,1/100)
    y_test_full[1+i*10:10+i*10,:,:]=slice_fading_input(y_temp[i+1,:,:], fading_weight,1/100)
end

fading_weight=Array([exp(lambda*(t[i]-time_range_test)) for i=1:length(t)]');

Threads.@threads for i in range(1,size(x_test)[1])
    x_test[i,:,:]=x_test[i,:,:].*fading_weight
end

y_test=y_test_full[:,1,size(y_test_full)[3]]

K=kernel_matrix(x_test,rbf)

weight2=ridge_regression_kernel(K, y_test, 0.1)

K_star=kernel_matrix(x_test,x_train,rbf)
K_star2=kernel_matrix(x_test,x_test,rbf)

error=sum(abs.(y_test-K_star2*weight2))/num_outputs_test

In [ ]:
x_scatter,y_scatter=scattering_trans(x,y)

amplification=1
lambda_scatter=1
fading_weight_scatter=Array([exp(lambda_scatter*(t[i]-time_range)) for i=1:length(t)]')

fading_weight_scatter=fading_weight_scatter/(sum(fading_weight_scatter.^2)*dt)^0.5*amplification;

x_train_scatter=Array{Float64,3}(undef,(num_outputs,input_size,num_points))
Threads.@threads for i in range(1,size(x)[1])
    x_train_scatter[i,:,:]=x_scatter[i,:,:].*fading_weight_scatter
end

y_train_scatter=y_scatter[:,1,size(y_scatter)[3]]

rbf=rbf_kernel_signal_gen(sigma,dt)
# beta=0.001
# alpha=0.004

alpha=0.0000000035
beta=0.1
cone=cone_kernel_signal_gen(alpha,beta,dt)
gain_schedule_kernel=gain_schedule_kernel_gen3(50,dt,num_points)
#gain_schedule_kernel=gain_schedule_kernel_gen(100,dt)
liner=linear_kernel_signal_gen(dt)

rbf_I=rbf_I_kernel_signal_gen(sigma,dt,num_points,alpha,beta)



K=kernel_matrix(x_train_scatter,cone)


weight_scatter=ridge_regression_kernel(K, y_train_scatter,5000000)



error=sum(abs.(y_train_scatter-K*weight_scatter))/size(y)[1]

println(error)
#print((sum((weight_scatter.*x_train_scatter).^2)*dt)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5)
println(((weight_scatter'*K*weight_scatter)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5))
println((weight_scatter'*K*weight_scatter)^0.5*(beta+alpha*sum(fading_weight_scatter.^2)*dt)^0.5)
println(((weight_scatter'*K*weight_scatter)^0.5*(sum(fading_weight_scatter.^2)*dt)^0.5)*alpha*2^0.5)
println(alpha*sum(abs.(weight_scatter))*(sum(fading_weight.^2)*dt)^0.5)

In [ ]:
plot([y_test_full[11,:,:]' y_test_full[12,:,: ]' ])

In [ ]:
x_demo=slice_fading_input(x_train[1,:,:], fading_weight,1)
plot([x_demo[997,:,:]' x_demo[1000,:,:]' ])

In [ ]:

x_demo=slice_fading_input(x[500,:,:], fading_weight,1)


K_star=kernel_matrix(x_demo,x_train,cone)

#K_star2=kernel_matrix(x_demo,x_test,rbf)
#K_star_exp=kernel_matrix(x_demo,x_train,exp_kernel)
#K_star_sigmoid=kernel_matrix(x_demo,x_train,sigmoid)
y_demo=K_star*weight_cone;


#y_demo=K_star_sigmoid*weight_sigmoid;


plot([y_demo[1:1000] y[500,:,1:1000]' ])




In [ ]:
tsample,x,y=Signal_output_step(sys,[[0,1,2]],x0, [2],[100],1000,1,10)
print(typeof(x))
fading_weight=Array([exp(1*(tsample[i]-2)) for i=1:length(tsample)])';
print(typeof(fading_weight))
print(size(fading_weight))
#slice_fading_input(x,fading_weight)
# basis=basis_fun2(3,5,1000,1)
print(size(x))
# print(size(x[1,:,:]))
# print(size(basis))
# map2Rn(basis, x)

In [ ]:

# print(typeof(fading_weight))
# print(size(fading_weight))
#slice_fading_input(x,fading_weight)
# basis=basis_fun2(3,5,1000,1)
# print(size(x[1,:,:]))
# print(size(basis))
# map2Rn(basis, x)
#print(x)

In [ ]:
typeof(x[1,:])

In [ ]:
tsample,x,y=Signal_output_step(sys,[0,1,2], 1000, x0, 2, 100, 50)


In [ ]:
print(typeof(y))
plot([y[3,:,:]' x[3,:,:]'])

In [ ]:
a=[1 2 3;4 5 6]
a.*[1, 2, 3]'
print(a.*[1, 2, 3]')


In [ ]:
reshape([[1,2] [3,4]], (4))